Re-analysis of existing dumps. **Accelerator: None** — this costs no GPU quota.

Add Input → Your Work → Notebook Output → the run notebook, so its `results/` is attached.

In [ ]:
REPO_URL = "https://github.com/Splestule/candidate_reranker.git"
BRANCH = "main"

In [ ]:
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/candidate_reranker")
if not (CODE / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)],
                   check=True)

sys.path.insert(0, str(CODE / "src"))
import kaggle_env as K

COMMIT = K.sync(REPO_URL, BRANCH)     # rerun this cell after every push
env = K.prepare_cpu(COMMIT)
DUMPS = K.find_dumps()

Tune on dev-clean. `gamma` is the new knob: 1.0 is one vote per candidate, 0.0 is one vote per cluster of near-identical candidates.

In [ ]:
K.run(env, "tune.py",
      "--dev", DUMPS["dev-clean"],
      "--test", DUMPS["test-clean"],
      "--json", env.results / f"tune-{COMMIT}.json")

Measure on all three test sets with the tuned parameters, with paired bootstrap intervals.

In [ ]:
import json

tuned = json.load(open(env.results / f"tune-{COMMIT}.json"))
print({k: tuned[k] for k in ("lambda", "alpha", "eps_conf", "gamma")})

for tag in ["test-clean", "test-other", "dialects"]:
    print("\n" + "#" * 74 + f"\n# {tag}\n" + "#" * 74)
    K.run(env, "analyze_compose.py", DUMPS[tag],
          "--alpha", tuned["alpha"], "--eps_conf", tuned["eps_conf"],
          "--gamma", tuned["gamma"],
          "--json", env.results / f"compose-{tag}-{COMMIT}.json")

What the cluster weighting is worth on its own: the same run at `gamma = 1.0`, everything else identical.

In [ ]:
for tag in ["test-clean", "test-other", "dialects"]:
    print("\n" + "#" * 74 + f"\n# {tag}  ·  gamma 1.0\n" + "#" * 74)
    K.run(env, "analyze_compose.py", DUMPS[tag],
          "--alpha", tuned["alpha"], "--eps_conf", tuned["eps_conf"], "--gamma", 1.0,
          "--n_boot", 1000,
          "--json", env.results / f"compose-{tag}-gamma1-{COMMIT}.json")

Copy the small result files into the repo checkout so they can be committed. Download them from the notebook output, or push from a machine with credentials.

In [ ]:
import shutil

keep = CODE / "results"
keep.mkdir(exist_ok=True)
for p in sorted(env.results.glob("*.json")):
    shutil.copy(p, keep / p.name)
    print(keep / p.name)